# Pose Estimator Test
This file is a test script of MediaPipe's Pose Estimator on pre-recorded video.

In [1]:
import numpy as np
import pandas as pd
import cv2
# import mediapipe as mp


import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles


In [2]:
ROOT = "D:/QEVD-FIT-300k/"
VIDEO_IDS = [
    "00001627",
    "00211894",
    "00297790",
    "00297803",
    "00297879",
    "00297885",
]

In [5]:
def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

# STEP 2: Create an PoseLandmarker object.
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True)
detector = vision.PoseLandmarker.create_from_options(options)


# # STEP 3: Load the input image.
# image = mp.Image.create_from_file("yoga.png")

# # STEP 4: Detect pose landmarks from the input image.
# detection_result = detector.detect(image)

# # STEP 5: Process the detection result. In this case, visualize it.
# annotated_image = draw_landmarks_on_image(image.numpy_view(), detection_result)
# output_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
# cv2.imwrite("output.png", output_image)
# # cv2_imshow()

In [15]:
for video_id in VIDEO_IDS:
    filepath = ROOT + video_id + ".mp4"
    cap = cv2.VideoCapture(filepath)

    fourcc = cv2.VideoWriter_fourcc(*"MP4V")
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_video = cv2.VideoWriter(f"out/pose{video_id}.mp4", fourcc, fps, (frame_width, frame_height))

    while cap.isOpened():
        _, frame = cap.read()
        try:
            frame_number = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

            detection_result = detector.detect(mp_image)
            annotated_image = draw_landmarks_on_image(mp_image.numpy_view(), detection_result)
            output_frame = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
            output_video.write(output_frame)
        except:
            break
    
    cap.release()
    output_video.release()
    cv2.destroyAllWindows()